In [1]:
import torch
from torch import nn


# Implement MHA

In [2]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, context_length):
        super().__init__()
        assert d_model % n_heads == 0, 'Error: n_heads is not divisible by d_model!'
        
        self.d_model = d_model
        self.d_h = d_model // n_heads
        self.n_heads = n_heads
        self.context_len = context_length
        
        self.W_key = nn.Linear(self.d_model, self.d_model)
        self.W_query = nn.Linear(self.d_model, self.d_model)
        self.W_value = nn.Linear(self.d_model, self.d_model)
        self.W_out = nn.Linear(self.d_model, self.d_model)

    def forward(self, x):
        batch_size, seq_len = x.shape[0], x.shape[1]
        
        k = self.W_key(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2) # (b, n_heads, seq_len, d_h)
        q = self.W_query(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2)
        v = self.W_value(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2)

        kdotq = k @ q.transpose(-1,-2) / (self.d_h ** 0.5) # (b, n_heads, seq_len, seq_len)

        if seq_len <= self.context_len:
            mask = (torch.triu(torch.ones((seq_len, seq_len)), 1) == 1)
        else:
            ones = torch.ones((seq_len,seq_len))
            mask = (torch.tril(ones,-self.context_len) + torch.triu(ones, 1) == 1)
        masked_kdotq = kdotq.masked_fill_(mask, -1e9)

        att_scores = torch.softmax(masked_kdotq, dim=-1)
        out = att_scores @ v  # (b, n_heads, seq_len, d_h)
        out = out.transpose(1,2) # (b, seq_len, n_heads, d_h)
        out = out.contiguous().view(batch_size, seq_len, self.d_model) 
        out = self.W_out(out)

        return out, att_scores
    

## tests

In [4]:
def test_long_context_window():
    bs, seq_len, d_model, n_heads = 8, 12, 64, 4
    x = torch.rand((bs, seq_len, d_model))
    
    mha = MultiHeadAttention(d_model, n_heads, 65536)
    out, att_scores = mha(x)
    
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [5]:
test_long_context_window()

torch.Size([8, 12, 64]) torch.Size([8, 4, 12, 12])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.4511, 0.5489, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.3042, 0.3614, 0.3343, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.2554, 0.2668, 0.2422, 0.2356, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1931, 0.2198, 0.1976, 0.1813, 0.2082, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1563, 0.1806, 0.1697, 0.1538, 0.1778, 0.1617, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1362, 0.1579, 0.1372, 0.1320, 0.1455, 0.1441, 0.1471, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1176, 0.1338, 0.1314, 0.1173, 0.1306, 0.1162, 0.1310, 0.1222, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1027, 0.1189, 0.11

In [6]:
def test_short_context_window():
    bs, seq_len, d_model, n_heads = 8, 12, 64, 4
    x = torch.rand((bs, seq_len, d_model))
    
    mha = MultiHeadAttention(d_model, n_heads, 7)
    out, att_scores = mha(x)
    
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [7]:
test_short_context_window()

torch.Size([8, 12, 64]) torch.Size([8, 4, 12, 12])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.5356, 0.4644, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.3499, 0.3181, 0.3319, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.2536, 0.2491, 0.2590, 0.2383, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.2144, 0.1922, 0.2045, 0.1834, 0.2055, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1822, 0.1639, 0.1792, 0.1517, 0.1655, 0.1575, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1511, 0.1400, 0.1479, 0.1346, 0.1452, 0.1437, 0.1375, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.1361, 0.1385, 0.1382, 0.1422, 0.1579, 0.1352, 0.1519, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.14

# Implement MHA with cached KV

In [3]:
class MultiHeadAttentionKVCache(nn.Module):
    def __init__(self, d_model, n_heads, context_length):
        super().__init__()
        assert d_model % n_heads == 0, 'Error: n_heads is not divisible by d_model!'
        
        self.d_model = d_model
        self.d_h = d_model // n_heads
        self.n_heads = n_heads
        self.context_len = context_length
        
        self.W_key = nn.Linear(self.d_model, self.d_model)
        self.W_query = nn.Linear(self.d_model, self.d_model)
        self.W_value = nn.Linear(self.d_model, self.d_model)
        self.W_out = nn.Linear(self.d_model, self.d_model)

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

    def forward(self, x, use_cache=False):
        batch_size, seq_len = x.shape[0], x.shape[1]
        
        q = self.W_query(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2)
        k_new = self.W_key(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2) # (b, n_heads, seq_len, d_h)
        v_new = self.W_value(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2)

        if use_cache:
            if self.cache_k is not None:
                self.cache_k = torch.concat([self.cache_k, k_new], dim=2)
                self.cache_v = torch.concat([self.cache_v, v_new], dim=2)
            else:
                self.cache_k, self.cache_v = k_new, v_new
            k, v = self.cache_k, self.cache_v          # (b, n_heads, cached_len, d_h)
            
        else:
            k, v = k_new, v_new
            self.cache_k, self.cache_v = None, None
            
        cached_len = k.shape[2]
        kdotq = q @ k.transpose(-1,-2) / (self.d_h ** 0.5) # (b, n_heads, seq_len, cached_len)

        upper_mask = (torch.arange(cached_len-seq_len-self.context_len,cached_len-self.context_len).unsqueeze(-1) >= torch.arange(cached_len).unsqueeze(0)) 
        lower_mask = (torch.arange(cached_len-seq_len,cached_len).unsqueeze(-1) < torch.arange(cached_len).unsqueeze(0))
        mask = upper_mask + lower_mask
        masked_kdotq = kdotq.masked_fill_(mask, -1e9)  # (b, n_heads, seq_len, cached_len)

        att_scores = torch.softmax(masked_kdotq, dim=-1)
        out = att_scores @ v  # (b, n_heads, seq_len, d_h)
        out = out.transpose(1,2) # (b, seq_len, n_heads, d_h)
        out = out.contiguous().view(batch_size, seq_len, self.d_model) 
        out = self.W_out(out)

        return out, att_scores
        
    def reset_kvcache(self):
        self.cache_k, self.cache_v = None, None

## tests

In [8]:
def test_kvcache_nocache():
    bs, seq_len, d_model, n_heads = 8, 12, 64, 4
    x = torch.rand((bs, seq_len, d_model))
    
    mha = MultiHeadAttentionKVCache(d_model, n_heads, 65536)
    out, att_scores = mha(x)
    
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [10]:
test_kvcache_nocache()

torch.Size([8, 12, 64]) torch.Size([8, 4, 12, 12])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.5036, 0.4964, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.3179, 0.3384, 0.3438, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.2460, 0.2491, 0.2557, 0.2492, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1953, 0.2015, 0.2053, 0.1982, 0.1997, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1590, 0.1633, 0.1705, 0.1748, 0.1641, 0.1683, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1370, 0.1414, 0.1425, 0.1450, 0.1398, 0.1474, 0.1469, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1193, 0.1202, 0.1228, 0.1233, 0.1288, 0.1255, 0.1365, 0.1236, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1065, 0.1038, 0.11

In [11]:
def test_kvcache_nocache_longcontext():
    bs, seq_len, d_model, n_heads = 8, 12, 64, 4
    x = torch.rand((bs, seq_len, d_model))

    context_window = 256
    mha = MultiHeadAttentionKVCache(d_model, n_heads, context_window)
    mha.cache_k = torch.rand((bs, n_heads, 3, d_model//n_heads))
    mha.cache_v = torch.rand((bs, n_heads, 3, d_model//n_heads))
    out, att_scores = mha(x, use_cache=True)
    
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [12]:
test_kvcache_nocache()

torch.Size([8, 12, 64]) torch.Size([8, 4, 12, 12])
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.4896, 0.5104, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.3236, 0.3326, 0.3438, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.2435, 0.2501, 0.2415, 0.2650, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1942, 0.1981, 0.2046, 0.2033, 0.1998, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1610, 0.1619, 0.1807, 0.1695, 0.1592, 0.1677, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1411, 0.1376, 0.1494, 0.1431, 0.1398, 0.1459, 0.1431, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1199, 0.1226, 0.1275, 0.1310, 0.1224, 0.1206, 0.1212, 0.1348, 0.0000,
         0.0000, 0.0000, 0.0000],
        [0.1058, 0.1095, 0.11

In [13]:
def test_kvcache_nocache_shortcontext():
    bs, seq_len, d_model, n_heads = 8, 3, 64, 4
    x = torch.rand((bs, seq_len, d_model))

    context_window = 8
    mha = MultiHeadAttentionKVCache(d_model, n_heads, context_window)
    mha.cache_k = torch.rand((bs, n_heads, 8, d_model//n_heads))
    mha.cache_v = torch.rand((bs, n_heads, 8, d_model//n_heads))
    out, att_scores = mha(x, use_cache=True)
    
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [14]:
test_kvcache_nocache_shortcontext()

torch.Size([8, 3, 64]) torch.Size([8, 4, 3, 11])
tensor([[0.0000, 0.1193, 0.1170, 0.1471, 0.1442, 0.1377, 0.1137, 0.1412, 0.0799,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.1228, 0.1713, 0.1386, 0.1236, 0.1271, 0.1294, 0.0857,
         0.1016, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.1462, 0.1235, 0.1347, 0.1083, 0.1253, 0.1079,
         0.1287, 0.1254]], grad_fn=<SelectBackward0>)


# Implement GQA with KV cache

In [24]:
class GroupQueryAttentionKVCache(nn.Module):
    def __init__(self, d_model, n_heads, context_length, n_groups):
        super().__init__()
        assert d_model % n_heads == 0, 'Error: d_model is not divisible by n_heads!'
        assert n_heads % n_groups == 0, 'Error: n_heads is not divisible by n_groups!'
        
        self.d_model = d_model
        self.d_h = d_model // n_heads
        self.n_heads = n_heads
        self.context_len = context_length
        self.n_groups = n_groups
        self.group_size = n_heads // n_groups
            
        self.W_query = nn.Linear(self.d_model, self.d_model)
        self.W_key = nn.Linear(self.d_model, self.n_groups * self.d_h)
        self.W_value = nn.Linear(self.d_model, self.n_groups * self.d_h)
        self.W_out = nn.Linear(self.d_model, self.d_model)

        self.register_buffer("cache_k", None, persistent=False)
        self.register_buffer("cache_v", None, persistent=False)

    def forward(self, x, use_cache=False):
        batch_size, seq_len = x.shape[0], x.shape[1]
        
        q = self.W_query(x).view(batch_size, seq_len, self.n_heads, self.d_h).transpose(1,2) # (b, n_heads, group_size, seq_len, d_h)
        
        k_new = self.W_key(x).view(batch_size, seq_len, self.n_groups, self.d_h).transpose(1,2) # (b, n_groups, seq_len, d_h)
        v_new = self.W_value(x).view(batch_size, seq_len, self.n_groups, self.d_h).transpose(1,2)

        if use_cache:
            if self.cache_k is not None:
                self.cache_k = torch.concat([self.cache_k, k_new], dim=2)
                self.cache_v = torch.concat([self.cache_v, v_new], dim=2)
            else:
                self.cache_k, self.cache_v = k_new, v_new
            k_bygroup, v_bygroup = self.cache_k, self.cache_v          # (b, n_groups, cached_len, d_h)
            
        else:
            k_bygroup, v_bygroup = k_new, v_new
            self.cache_k, self.cache_v = None, None
            
        k = k_bygroup.repeat_interleave(self.group_size, dim=1) # (b, n_heads, cached_len, d_h)
        v = v_bygroup.repeat_interleave(self.group_size, dim=1) # (b, n_heads, cached_len, d_h)
        
        cached_len = k.shape[2]
        kdotq = q @ k.transpose(-1,-2) / (self.d_h ** 0.5) # (b, n_heads, seq_len, cached_len)

        upper_mask = (torch.arange(cached_len-seq_len-self.context_len,cached_len-self.context_len).unsqueeze(-1) >= torch.arange(cached_len).unsqueeze(0)) 
        lower_mask = (torch.arange(cached_len-seq_len,cached_len).unsqueeze(-1) < torch.arange(cached_len).unsqueeze(0))
        mask = upper_mask + lower_mask
        masked_kdotq = kdotq.masked_fill_(mask, -1e9)  # (b, n_heads, seq_len, cached_len)

        att_scores = torch.softmax(masked_kdotq, dim=-1)
        out = att_scores @ v  # (b, n_heads, seq_len, d_h)
        out = out.transpose(1,2) # (b, seq_len, n_heads, d_h)
        out = out.contiguous().view(batch_size, seq_len, self.d_model) 
        out = self.W_out(out)

        return out, att_scores
        
    def reset_kvcache(self):
        self.cache_k, self.cache_v = None, None

## test

In [40]:
def test_gqa():
    bs, seq_len, d_model, n_heads, n_groups = 8, 12, 64, 16, 8
    x = torch.rand((bs, seq_len, d_model))
    cache_k = torch.rand((bs, n_groups, 10, d_model//n_heads))
    cache_v = torch.rand((bs, n_groups, 10, d_model//n_heads))

    context_window = 8
    
    gqa = GroupQueryAttentionKVCache(d_model, n_heads, context_window, n_groups)
    gqa.cache_k = cache_k
    gqa.cache_v = cache_v
    out, att_scores = gqa(x, use_cache=True)
        
    assert out.shape == x.shape, "output shape does not match input shape"
    
    print(out.shape, att_scores.shape)
    print(att_scores[0][0])

In [41]:
test_gqa()

torch.Size([8, 12, 64]) torch.Size([8, 16, 12, 22])
tensor([[0.0000, 0.0000, 0.0000, 0.1326, 0.1169, 0.1287, 0.1411, 0.1185, 0.1036,
         0.1344, 0.1242, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.1098, 0.1295, 0.1261, 0.1136, 0.1111,
         0.1252, 0.1364, 0.1484, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1391, 0.1403, 0.1125, 0.1060,
         0.1411, 0.1250, 0.1257, 0.1103, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.1423, 0.1192, 0.1079,
         0.1374, 0.1244, 0.1229, 0.1130, 0.1331, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0966, 0.0985,
         0.1340, 0.1337, 0.1327, 0.1152, 0.1532, 0.